1. Data load karo (Day 34 ka cleaned version):

In [1]:
import pandas as pd

df = pd.read_csv(
    '../data/processed/step3_all_valid_transactions.csv',
    dtype={'Invoice': str, 'StockCode': str}
)
print(df.dtypes)

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object


Notice : Agar tumne CSV dobara load kiya (bina parse_dates), to InvoiceDate phir se string (object) ban gayi hoga — kyunki CSV mein sab kuch text ke roop mein save hota hai.

2. pd.to_datetime() — string ko datetime mein convert karna:

In [2]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)   # datetime64[ns] aana chahiye
print(df['InvoiceDate'].head())

datetime64[us]
0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
Name: InvoiceDate, dtype: datetime64[us]


3. Datetime column se useful parts nikalna (.dt accessor):

In [3]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()   # Monday, Tuesday, etc.
df['Hour'] = df['InvoiceDate'].dt.hour

print(df[['InvoiceDate', 'Year', 'Month', 'DayOfWeek', 'Hour']].head())

          InvoiceDate  Year  Month DayOfWeek  Hour
0 2009-12-01 07:45:00  2009     12   Tuesday     7
1 2009-12-01 07:45:00  2009     12   Tuesday     7
2 2009-12-01 07:45:00  2009     12   Tuesday     7
3 2009-12-01 07:45:00  2009     12   Tuesday     7
4 2009-12-01 07:45:00  2009     12   Tuesday     7


Real use-case: Isse business insights milte hain — jaise "kaunse din/hour mein sabse zyada orders aate hain" (Month 3 ke EDA mein kaam aayega).

4. Numeric columns ka type check aur zaroorat pade to convert karna:

In [4]:
print(df[['Quantity', 'Price']].dtypes)

# Agar kisi wajah se yeh 'object' (string) ban jaaye, to convert karna:
# df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
# 'coerce' matlab agar convert na ho paaye to NaN bana do (crash nahi hoga)

Quantity      int64
Price       float64
dtype: object


5. Customer ID ko float se int mein convert karna (SQL/analysis mein cleaner dikhta hai):

In [5]:
print(df['Customer ID'].dtype)   # float64 (kyunki NaN tha pehle, isliye Pandas float rakhta hai)
df['Customer ID'] = df['Customer ID'].astype(int)
print(df['Customer ID'].dtype)
print(df['Customer ID'].head())

float64
int64
0    13085
1    13085
2    13085
3    13085
4    13085
Name: Customer ID, dtype: int64


Note: Jab tak NaN values thi, Pandas Customer ID ko float rakhta tha (kyunki int mein NaN store nahi ho sakta) — ab Day 32 mein NaN drop kar chuke ho, isliye ab safely int mein convert kar sakte ho.

6. Country ko category dtype mein convert karna (memory optimization — advanced tip):

In [6]:
print(f"Memory before: {df['Country'].memory_usage(deep=True)} bytes")
df['Country'] = df['Country'].astype('category')
print(f"Memory after: {df['Country'].memory_usage(deep=True)} bytes")

Memory before: 48535129 bytes
Memory after: 781968 bytes


Note: category dtype tab useful hai jab column mein limited unique values hon (jaise Country — sirf ~40 unique values, lakhon rows) — Pandas internally numbers use karta hai text ki jagah, memory bahut bachta hai.

7. Final dtype check aur save karna:

In [7]:
print(df.dtypes)
df.to_csv('../data/processed/step4_datatypes_fixed.csv', index=False)
print(f"Saved: {df.shape}")

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country              category
Year                    int32
Month                   int32
Day                     int32
DayOfWeek                 str
Hour                    int32
dtype: object
Saved: (779495, 13)


Practice questions:

1.DayOfWeek column use karke pata karo kaunsa din sabse zyada orders wala hai (value_counts()).

Ans: value_counts() descending order mein data deta hai, isliye pehli row wahi din hogi jahan sabse zyada transactions hue hain (Online Retail dataset mein aam taur par Thursday ya Wednesday sabse busy din hote hain).

In [10]:
# Har din ke total orders count karna
busiest_days = df['DayOfWeek'].value_counts()
print(busiest_days)

# Sabse zyada orders wala din (Top day)
print("\nSabse busy din:")
print(busiest_days.idxmax(), "->", busiest_days.max(), "orders")

DayOfWeek
Thursday     156012
Tuesday      134028
Wednesday    130782
Sunday       130141
Monday       124957
Friday       103175
Saturday        400
Name: count, dtype: int64

Sabse busy din:
Thursday -> 156012 orders


2.StockCode ko bhi category dtype mein convert karke memory savings check karo.

- Memory Optimization Tip: Kyunki StockCode ek string column hai aur isme unique values ke muqable total rows bohot zyada hoti hain, category dtype isko integers (codes) ke roop mein store karta hai, jisse RAM ki khapat dramatic tareeqe se kam ho jaati hai!

In [11]:
# Memory check karne se pehle
mem_before = df['StockCode'].memory_usage(deep=True)

# Category mein convert karna
df['StockCode'] = df['StockCode'].astype('category')

# Memory check karne ke baad
mem_after = df['StockCode'].memory_usage(deep=True)

print(f"Memory before: {mem_before:,} bytes")
print(f"Memory after: {mem_after:,} bytes")
print(f"Total Memory Saved: {mem_before - mem_after:,} bytes")

Memory before: 42,174,799 bytes
Memory after: 1,810,497 bytes
Total Memory Saved: 40,364,302 bytes
